### Tools and Agents

In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\antoc\AppData\Local\Temp\ipykernel_27956\3020703078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [4]:
tools=[wiki,arxiv]

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
documents=loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs=text_splitter.split_documents(documents)
vectordb = FAISS.from_documents(docs, OpenAIEmbeddings())
reteriver = vectordb.as_retriever()


In [8]:
reteriver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000273AECA8D10>, search_kwargs={})

In [9]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(reteriver, "langsmith-search", "Search any information about LangSmith")

In [10]:
retriever_tool.name

'langsmith-search'

In [11]:
tools = [wiki,arxiv,retriever_tool]

In [12]:
import openai
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
openai.api_key = os.getenv("OPENAI_API_KEY")

llm = ChatGroq(api_key=groq_api_key, model_name="openai/gpt-oss-20b")

In [13]:
# Prompt Template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [14]:
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(
    llm, tools, prompt
)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [15]:
from langchain_classic.agents import AgentExecutor
from langchain.agents import create_agent 
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    return_intermediate_steps=True
)

In [16]:
agent_executor.run({"input":"What is LangSmith?"})

C:\Users\antoc\AppData\Local\Temp\ipykernel_27956\3532863195.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent_executor.run({"input":"What is LangSmith?"})


[(ToolAgentAction(tool='langsmith-search', tool_input={'query': 'LangSmith'}, log="\nInvoking: `langsmith-search` with `{'query': 'LangSmith'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'reasoning_content': 'We should use langsmith-search to find info.', 'tool_calls': [{'index': 0, 'id': 'fc_e21b6fb6-56ee-4dcc-b205-e5e02e2ffd1f', 'function': {'arguments': '{"query":"LangSmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand'}, id='lc_run--01a15e20-817d-70b7-8e49-e4130620179b', tool_calls=[{'name': 'langsmith-search', 'args': {'query': 'LangSmith'}, 'id': 'fc_e21b6fb6-56ee-4dcc-b205-e5e02e2ffd1f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 36, 'total_tokens': 310, 'output_token_details': {'reasoning': 11}}, tool_call_chunks=

In [17]:
agent_executor.invoke({"input":"What is machine learning"})

{'input': 'What is machine learning',
 'output': '**Machine learning (ML)** is a subfield of artificial intelligence that focuses on building systems that can learn from data and improve their performance on a task without being explicitly programmed for every possible situation.\n\n---\n\n### Core Concepts\n\n| Concept | What it means | Example |\n|---------|---------------|---------|\n| **Data** | The raw information the system uses to learn. | Images of cats and dogs, sensor readings, customer purchase histories. |\n| **Model** | A mathematical representation that maps inputs to outputs. | A neural network that predicts house prices. |\n| **Training** | The process of adjusting the model’s parameters to reduce error on a training dataset. | Using gradient descent to minimize the difference between predicted and actual labels. |\n| **Loss function** | Quantifies how far the model’s predictions are from reality. | Mean squared error for regression, cross‑entropy for classification. |\